# OpenMHC Quickstart

End-to-end walkthrough of the public evaluation API:

1. Download the tiny dataset
2. Track 2 — define a trivial `Imputer` and run `evaluate_imputation`
3. Track 1 — define a trivial `Encoder` and run `evaluate_prediction`
4. Track 3 — define a trivial `Forecaster` and run `evaluate_forecasting`
5. Generate a paste-ready submission body

This notebook is the smallest end-to-end loop. It runs on CPU and finishes in a few minutes against the tiny version.

## 1. Install + download the dataset

If you haven't already:

```bash
pip install -e ..
```

Then download the XS dataset (~TBD MB).

In [ ]:
import os
from pathlib import Path

import openmhc

# Pick a destination, or set MHC_DATA_DIR before running this cell.
DATA_ROOT = Path(os.environ.get("MHC_DATA_DIR", "~/.cache/openmhc/data-xs")).expanduser()

# Download only if not already present; otherwise reuse the cached copy.
if not (DATA_ROOT / "dataset_version.json").exists():
    openmhc.download_dataset(version="xs", dest=DATA_ROOT)

# Make this the dataset root for every subsequent openmhc.* call.
os.environ["MHC_DATA_DIR"] = str(DATA_ROOT)
print(f"dataset is at: {DATA_ROOT}")

## 2. Define a trivial Imputer

Models implement the `Imputer` protocol — one method, no inheritance:
- `impute(data, observed_mask, target_mask)` — fill in artificially-masked positions

All setup (loading checkpoints, computing statistics from the training split, building per-user state) happens in `__init__`; the benchmark harness does not call any preparation method. Below we compute the per-channel global mean once from the train split via `openmhc.iter_train_data(version=VERSION)`, then use it as the fill value.

`VERSION` is set once and threaded through every public-API call. There is no auto-detection: the dataset root carries a `dataset_version.json` marker and the resolver insists the version you pass matches it.

In [ ]:
import numpy as np
import openmhc

# Match what was downloaded above. Switch to "full" once you have the full release.
VERSION = "xs"


class MeanImputer:
    """Predict the channel-wise mean for every masked position."""

    def __init__(self):
        # Compute per-channel mean from the official train split.
        data_sum = np.zeros(19, dtype=np.float64)
        data_count = np.zeros(19, dtype=np.float64)
        for data, mask in openmhc.iter_train_data(version=VERSION):
            obs = (mask > 0.5) & np.isfinite(data)
            data_sum += np.where(obs, data, 0.0).sum(axis=(0, 2))
            data_count += obs.sum(axis=(0, 2))
        self.means = (data_sum / np.maximum(data_count, 1)).astype(np.float32)

    def impute(
        self,
        data: np.ndarray,
        observed_mask: np.ndarray,
        target_mask: np.ndarray,
    ) -> np.ndarray:
        out = data.copy()
        for ch in range(19):
            target = target_mask[:, ch, :] == 1
            out[:, ch, :][target] = self.means[ch]
        return out.astype(np.float32)

## 3. Run imputation evaluation

`evaluate_imputation` runs your imputer against all 6 masking scenarios (random noise, temporal slices, sleep gaps, etc.) on the test split.

In [ ]:
results = openmhc.evaluate_imputation(MeanImputer(), version=VERSION)
results.summary()

## 4. Define a trivial Encoder

For Track 1 (outcome prediction) the protocol is just `encode(weekly_tensors) -> embeddings`. Here we mean-pool over the time axis.

In [ ]:
class MeanPoolEncoder:
    """Average over the 168 hours, take the 19 sensor channels."""

    def encode(self, weekly_tensors: np.ndarray) -> np.ndarray:
        # weekly_tensors: (B, 168, 38). Cols 0-18 sensors, 19-37 missingness.
        return weekly_tensors[:, :, :19].mean(axis=1).astype(np.float32)

## 5. Run prediction evaluation (Track 1)

`evaluate_prediction` extracts your embeddings, then fits a linear probe per task (logistic regression for binary, ordinal logistic for ordinal, ridge regression for continuous).

In [ ]:
# tasks="all" is fine on the XS dataset; restrict if you want a faster loop
results = openmhc.evaluate_prediction(
    MeanPoolEncoder(), version=VERSION, tasks=["BiologicalSex", "age"]
)
results.summary()

## 7. Generate a leaderboard submission

`results.to_submission_yaml(...)` produces a paste-ready body matching the textareas in the [submission issue template](https://github.com/AshleyLab/myheartcounts-dataset/issues/new?template=submission.yml).

For Track 2 imputation, skill scores and per-category subgroup scores are computed locally against the frozen LOCF baseline (`data/baselines/imputation_locf.json`). For Track 1 and Track 3, those fields are emitted as `—` and filled in by the maintainers from `raw_metrics` during ingestion.

In [ ]:
class LastValueForecaster:
    """Repeat the most recent observed value across the forecast horizon."""

    def predict(self, history: np.ndarray, horizon: int) -> np.ndarray:
        # history: (n_channels, history_length); returns (n_channels, horizon)
        last = np.nan_to_num(history[:, -1:], nan=0.0)
        return np.tile(last, (1, horizon)).astype(np.float32)


# max_samples keeps this fast on the tiny dataset; remove for full eval
forecast_results = openmhc.evaluate_forecasting(
    LastValueForecaster(), version=VERSION, forecasting_length=24, max_samples=20
)
forecast_results.summary()

## 6. Generate a leaderboard submission

`results.to_submission_yaml(...)` produces a paste-ready body matching the textareas in the [submission issue template](https://github.com/AshleyLab/myheartcounts-dataset/issues/new?template=submission.yml).

For Track 2 imputation, skill scores and per-category subgroup scores are computed locally against the frozen LOCF baseline (`data/baselines/imputation_locf.json`). For Track 1, those fields are emitted as `—` and filled in by the maintainers from `raw_metrics` during ingestion.

In [ ]:
imputation_results = openmhc.evaluate_imputation(MeanImputer(), version=VERSION)

submission_body = imputation_results.to_submission_yaml(
    method_name="MeanImputer (demo)",
    submitter_team="Stanford CS",
    paper_url="https://arxiv.org/abs/0000.00000",
    code_url="https://github.com/example/mean-imputer",
    method_category="Statistical / Classical baseline",
)
print(submission_body)

Once you have a body that looks right:

1. Open a [submission issue](https://github.com/AshleyLab/myheartcounts-dataset/issues/new?template=submission.yml).
2. Fill in the single-input fields (method name, paper URL, …) and paste the three textarea blocks above.
3. The HuggingFace Space ingests merged submissions and the public leaderboard at `https://myheartcounts.stanford.edu/benchmark` rebuilds automatically.

Submissions must follow the standard protocol — same split file, masking config, label-validity criterion as the paper. The submission template enforces required fields.